# 02 - Data Sanity Check

Run this after the metadata has been copied or pulled into `data/metadata/labeled_dataset.csv`. The notebook inspects columns, label balance, and the intended balanced subset strategy without modifying heavy data.

In [ ]:
REPO_DIR = "/content/graphcnn-federated-3d"
%cd {REPO_DIR}

In [ ]:
from pathlib import Path
import pandas as pd

from src.data.dataset import ShapeNetPointCloudDataset
from src.utils.config import load_config

config = load_config("configs/data.yaml")
metadata_path = Path(config["paths"]["metadata_csv"])
print(config)

In [ ]:
if metadata_path.exists():
    df = pd.read_csv(metadata_path)
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    display(df.head())
else:
    df = pd.DataFrame()
    print(f"Missing metadata: {metadata_path}")
    print("Upload labeled_dataset.csv to Google Drive and run the all-in-one Colab notebook, or copy it manually into data/metadata/.")

In [ ]:
if not df.empty:
    label_candidates = [c for c in df.columns if "label" in c.lower() or "class" in c.lower() or "category" in c.lower()]
    label_col = "label" if "label" in df.columns else (label_candidates[0] if label_candidates else None)
    print("Label column:", label_col)
    if label_col:
        counts = df[label_col].value_counts()
        print("Number of labels:", counts.shape[0])
        display(counts.head(30).rename_axis("label").reset_index(name="count"))
    else:
        print("Could not infer a label column. Rename or pass the correct label column in dataset code.")

In [ ]:
if not df.empty and 'label_col' in globals() and label_col:
    subset_classes = int(config["subset_classes"])
    samples_per_class = int(config["samples_per_class"])
    chosen_labels = df[label_col].value_counts().head(subset_classes).index.tolist()
    subset_preview = (
        df[df[label_col].isin(chosen_labels)]
        .groupby(label_col, group_keys=False)
        .head(samples_per_class)
        .reset_index(drop=True)
    )
    print("Chosen labels:", chosen_labels)
    print("Subset preview rows:", len(subset_preview))
    display(subset_preview[label_col].value_counts().rename_axis("label").reset_index(name="count"))
else:
    print("Subset preview skipped because metadata or labels are unavailable.")

In [ ]:
dataset = ShapeNetPointCloudDataset(metadata_csv=metadata_path)
dataset.summary()

In [ ]:
!pytest -q tests/test_dataset.py tests/test_shapes.py tests/test_fedavg.py tests/test_distillation.py